# 06 — BQ4: How Do Course Characteristics Affect Retention?

> **Notebook 06 of 7** | Learning Retention Analytics  
> Fourth business question analysis: comparing course design features and their relationship with student retention.

## Purpose and Scope

This notebook answers **BQ4: How do course characteristics affect retention?** — the fourth of five business questions driving this project.

Previous notebooks focused on student-level analysis: when students drop out (BQ1), which early behaviors predict dropout (BQ2), and whether demographics or behavior predicts outcome more strongly (BQ3). This notebook **shifts the unit of analysis from students to courses**.

**What this notebook covers:**
- Ranking the 7 OULAD modules by completion rate
- Exploring associations between course design features (length, assessment density, resource diversity) and retention
- Comparing engagement intensity across courses
- Identifying course-level patterns through a standardized design heatmap
- Formulating hypotheses about which course characteristics may support retention

**What this notebook does NOT do:**
- No inferential statistics. With only 7 modules, formal significance tests have no power. All analysis is descriptive and exploratory.
- No causal claims. Differences in retention could be driven by student selection, subject difficulty, or unmeasured factors — not course design alone.

**What comes next:**
- **Notebook 07** (`07_bq5_recommendations_synthesis.ipynb`): synthesizing BQ1–BQ4 findings into the top 3 actionable interventions (BQ5).

> **Methodological transferability:** Comparing course-level retention metrics parallels product-level or plan-level churn analysis in SaaS: which product tier retains best? Which plan characteristics (trial length, feature set, onboarding flow) correlate with lower churn? The approach — descriptive ranking, feature profiling, heatmap comparison — transfers directly.

## Table of Contents

1. [Environment Setup](#1.-Environment-Setup)
2. [The Analysis Dataset](#2.-The-Analysis-Dataset)
3. [Course Ranking by Completion](#3.-Course-Ranking-by-Completion)
4. [Course Design Features vs Completion](#4.-Course-Design-Features-vs-Completion)
5. [Engagement Intensity by Course](#5.-Engagement-Intensity-by-Course)
6. [Course Design Heatmap](#6.-Course-Design-Heatmap)
7. [Hypotheses and Limitations](#7.-Hypotheses-and-Limitations)
8. [Key Takeaways and Next Steps](#8.-Key-Takeaways-and-Next-Steps)

---

**Prerequisites:**
- The ETL pipeline must have been run: `python -m run_pipeline`
- The DuckDB database at `data/db/oulad.duckdb` must contain all 5 analytical views

**Dataset:** Open University Learning Analytics Dataset (OULAD) — ~32K students, 7 courses, complete behavioral clickstream. License: CC-BY 4.0.

## 1. Environment Setup

We configure imports, visualization defaults, and reusable helper functions.

**Technical notes for readers:**
- All database queries go through `src.db.connection.execute_query()` — the project's DB abstraction layer (ADR-003).
- BQ4's primary SQL query lives in `sql/queries/q_bq4_course_comparison.sql` and is loaded at runtime from disk. The query aggregates course-level metrics from `v_course_profile`, `v_engagement_daily`, and `v_engagement_early`.
- No statistical test functions are imported — with only 7 data points, all analysis is descriptive.
- Figures are saved to `reports/figures/` at 150 DPI.

In [ ]:
# --- Path setup ---
# Notebooks live in notebooks/ but project modules are at the project root.
# We search upward for pyproject.toml so the notebook works regardless of
# where the kernel is launched from (JupyterLab, VS Code, Cursor, repo root).
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    """Walk upward from start until we find pyproject.toml (the repo root marker)."""
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise RuntimeError(
        f"Could not locate project root: no pyproject.toml found in '{start}' "
        "or any parent directory. Run the notebook from within the repository."
    )


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# --- Standard library ---
import logging
import warnings

# --- Third-party ---
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# --- Project modules ---
from src.config import FIGURES_DIR, QUERIES_DIR
from src.db.connection import execute_query

# --- Configuration ---
warnings.filterwarnings('ignore', category=FutureWarning)
logging.basicConfig(level=logging.WARNING)

# --- Visualization defaults ---
sns.set_theme(style='whitegrid', font_scale=1.1)

PALETTE_OUTCOME = {
    'Pass': '#4C72B0',
    'Distinction': '#55A868',
    'Fail': '#C44E52',
    'Withdrawn': '#8172B3',
}
LABEL_COMPLETED = 'Completed'
LABEL_NOT_COMPLETED = 'Not completed'
PALETTE_BINARY = {1: '#55A868', 0: '#C44E52'}
LABEL_BINARY = {1: LABEL_COMPLETED, 0: LABEL_NOT_COMPLETED}
PALETTE_BINARY_LABELS = {LABEL_COMPLETED: '#55A868', LABEL_NOT_COMPLETED: '#C44E52'}
PALETTE_SEQUENTIAL = 'YlOrRd'

# Course-level palette: one color per module for consistent identification.
# 7 OULAD modules use tab10 for maximum visual distinction.
_MODULE_ORDER = ['AAA', 'BBB', 'CCC', 'DDD', 'EEE', 'FFF', 'GGG']
_TAB10 = plt.cm.tab10.colors
PALETTE_COURSE = {m: _TAB10[i] for i, m in enumerate(_MODULE_ORDER)}

# Shared axis labels — defined as constants to avoid
# duplicated string literals flagged by static analysis
LABEL_COMPLETION_RATE = 'Completion rate (%)'
LABEL_WITHDRAWAL_RATE = 'Withdrawal rate (%)'
LABEL_NUM_ENROLLMENTS = 'Number of enrollments'

FIG_DPI = 150
FIG_SIZE = (10, 6)
FIG_SIZE_WIDE = (16, 5)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name: str) -> None:
    """Save figure to reports/figures/ with consistent settings."""
    path = FIGURES_DIR / f'{name}.png'
    fig.savefig(path, dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
    print(f'  Saved: {path}')


# --- Load BQ4 query from SQL file ---
_bq4_sql_path = QUERIES_DIR / 'q_bq4_course_comparison.sql'
BQ4_SQL = _bq4_sql_path.read_text(encoding='utf-8')
print(f'Loaded BQ4 query from: {_bq4_sql_path.name} ({len(BQ4_SQL):,} chars)')

# --- Prerequisite check ---
# BQ4 query uses three views: v_course_profile (FROM), v_engagement_daily
# and v_engagement_early (scalar subqueries for engagement intensity).
try:
    _check_cp = execute_query('SELECT COUNT(*) AS n FROM v_course_profile')
    _check_daily = execute_query('SELECT COUNT(*) AS n FROM v_engagement_daily')
    _check_early = execute_query('SELECT COUNT(*) AS n FROM v_engagement_early')
    _n_cp = _check_cp['n'].iloc[0]
    _n_daily = _check_daily['n'].iloc[0]
    _n_early = _check_early['n'].iloc[0]
    if _n_cp == 0 or _n_daily == 0 or _n_early == 0:
        raise RuntimeError('One or more views are empty')
    print('Database OK')
    print(f'  v_course_profile:    {_n_cp:>12,} rows')
    print(f'  v_engagement_daily:  {_n_daily:>12,} rows')
    print(f'  v_engagement_early:  {_n_early:>12,} rows')
except Exception as exc:
    raise RuntimeError(
        'Cannot query analytical views. '
        "Run 'python -m run_pipeline' first to populate the database."
    ) from exc

## 2. The Analysis Dataset

The BQ4 query (`q_bq4_course_comparison.sql`) aggregates course-level metrics from `v_course_profile`, enriched with engagement intensity from `v_engagement_daily` and `v_engagement_early`. Each row represents **one module** (aggregated across all its presentations):

| Category | Columns | Description |
|----------|---------|-------------|
| **Scale** | n_presentations, total_enrolled | How many cohorts and students |
| **Outcomes** | avg_completion_rate_pct, avg_withdrawal_rate_pct | Retention performance |
| **Course design** | avg_course_length_days, avg_n_assessments, avg_assessment_density, avg_n_vle_resources, avg_n_activity_types | Characteristics the course designer controls |
| **Engagement** | avg_clicks_per_student_day, median_early_clicks | How intensely students engage with the course |

**Key design decision:** Aggregating by `code_module` (not `code_module + code_presentation`) gives a stable per-course view. Individual presentations can vary, but the module's design (assessments, resources, structure) is the controllable variable.

**Critical caveat:** With only **7 data points** (one per module), this analysis is entirely descriptive. No statistical test can reliably detect effects at this sample size. Patterns are hypotheses, not conclusions.

In [ ]:
# --- Load BQ4 analysis dataset ---
df_bq4 = execute_query(BQ4_SQL)

modules_str = ', '.join(df_bq4['code_module'].tolist())
total_enrolled = df_bq4['total_enrolled'].sum()
total_presentations = df_bq4['n_presentations'].sum()

print(f'BQ4 dataset: {len(df_bq4)} rows (one per module) x {df_bq4.shape[1]} columns')
print(f'Modules: {modules_str}')
print(f'Total enrollments across all modules: {total_enrolled:,}')
print(f'Total presentations: {total_presentations}')

# --- Feature column constants ---
DESIGN_FEATURES = [
    'avg_course_length_days', 'avg_n_assessments',
    'avg_assessment_density', 'avg_n_vle_resources', 'avg_n_activity_types',
]
DESIGN_LABELS = {
    'avg_course_length_days': 'Course length (days)',
    'avg_n_assessments': 'Number of assessments',
    'avg_assessment_density': 'Assessment density (per 30d)',
    'avg_n_vle_resources': 'VLE resources',
    'avg_n_activity_types': 'Activity types',
}

ENGAGEMENT_FEATURES = ['avg_clicks_per_student_day', 'median_early_clicks']
ENGAGEMENT_LABELS = {
    'avg_clicks_per_student_day': 'Avg clicks per student-day',
    'median_early_clicks': 'Median early clicks (first 28d)',
}

# --- Missingness check ---
n_nulls = df_bq4.isnull().sum()
if n_nulls.any():
    print()
    print('=== Null Values Detected ===')
    print(n_nulls[n_nulls > 0].to_string())
else:
    print()
    print('No null values — all 7 modules have complete data.')

# --- Full dataset overview ---
print()
print('=== Course Comparison Dataset (sorted by completion rate) ===')
print()
print(df_bq4.to_string(index=False))

> **First look:** The 7 OULAD modules span a wide range of completion rates and course designs. Some modules are shorter with fewer assessments; others are longer with more resources and higher assessment density. The question is whether these design differences relate to retention performance.
>
> The dataset is deliberately small — one row per module — which means every observation matters and no outlier can be dismissed. All patterns we identify are hypotheses that would need larger-scale validation.

## 3. Course Ranking by Completion

Before exploring design features, we establish the baseline: which courses retain students best?

The completion rate is averaged across all presentations of each module. This smooths out cohort-specific variation and focuses on the module's inherent retention characteristics.

In [ ]:
# --- Course completion ranking ---
# Horizontal bar chart sorted by avg_completion_rate_pct.
# Module colors from PALETTE_COURSE provide visual consistency across figures.
df_ranked = df_bq4.sort_values(
    'avg_completion_rate_pct', ascending=True
).reset_index(drop=True)

fig, ax = plt.subplots(figsize=FIG_SIZE)

y_pos = np.arange(len(df_ranked))
colors = [PALETTE_COURSE[m] for m in df_ranked['code_module']]

ax.barh(y_pos, df_ranked['avg_completion_rate_pct'], color=colors, edgecolor='white')

# Annotate each bar with completion rate and enrollment volume
for i, (_, row) in enumerate(df_ranked.iterrows()):
    rate = row['avg_completion_rate_pct']
    enrolled = int(row['total_enrolled'])
    ax.text(
        rate + 0.8, i,
        f'{rate:.1f}%  (n={enrolled:,})',
        va='center', fontsize=9, color='#333333',
    )

# Enrollment-weighted average across modules as reference line
overall_rate = (
    (df_bq4['avg_completion_rate_pct'] * df_bq4['total_enrolled']).sum()
    / df_bq4['total_enrolled'].sum()
)
ax.axvline(
    x=overall_rate, color='gray', linestyle='--', linewidth=1,
    label=f'Weighted avg: {overall_rate:.1f}%',
)

ax.set_yticks(y_pos)
ax.set_yticklabels(df_ranked['code_module'])
ax.set_xlabel(LABEL_COMPLETION_RATE)
ax.set_title('Course Completion Rate Ranking\n(averaged across presentations)')
ax.legend(loc='lower right')
ax.set_xlim(0, 100)
sns.despine()
fig.tight_layout()
save_fig(fig, '06_course_completion_ranking')
plt.show()

# --- Quantify the spread ---
best = df_ranked.iloc[-1]
worst = df_ranked.iloc[0]
gap = best['avg_completion_rate_pct'] - worst['avg_completion_rate_pct']
best_rate = best['avg_completion_rate_pct']
worst_rate = worst['avg_completion_rate_pct']
best_module = best['code_module']
worst_module = worst['code_module']
print()
print(f'Completion rate range: {worst_rate:.1f}% ({worst_module}) '
      f'to {best_rate:.1f}% ({best_module}) — gap of {gap:.1f} pp')

> **Key finding:** The completion rate gap between the best- and worst-performing modules is substantial. This variation is not random — it persists across multiple presentations of the same module, suggesting that something about the course itself (design, subject difficulty, student selection) drives the difference.
>
> **Note on enrollment volume:** The annotation shows total enrollment for each module. Larger enrollment doesn't necessarily correlate with higher or lower completion — the pattern is more nuanced, and we'll explore it through design features next.

## 4. Course Design Features vs Completion

Do course design characteristics correlate with retention? We examine two key design dimensions:

- **Course length** (days): Does duration affect whether students stay enrolled?
- **Assessment density** (assessments per 30 days): Does more frequent assessment help or hinder retention?

Each scatter plot shows the 7 modules positioned by their design feature (x-axis) and completion rate (y-axis). Point size is proportional to total enrollment for additional context.

**Reminder:** With n=7, we are looking for *visual patterns*, not statistical relationships. Any observed trend is a hypothesis to be tested with more data.

In [ ]:
# --- Course design features vs completion rate ---
# 1x2 scatter panel: course length and assessment density vs completion.
# Bubble size proportional to enrollment volume for context.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=FIG_SIZE_WIDE)

# Scale enrollment to bubble sizes (100–600 pixel range)
enrolled = df_bq4['total_enrolled'].values
size_scale = 100 + 500 * (enrolled - enrolled.min()) / max(enrolled.max() - enrolled.min(), 1)

# --- Left: course length vs completion ---
for i, (_, row) in enumerate(df_bq4.iterrows()):
    ax1.scatter(
        row['avg_course_length_days'], row['avg_completion_rate_pct'],
        color=PALETTE_COURSE[row['code_module']],
        s=size_scale[i], edgecolor='white', linewidth=1.5, zorder=3,
    )
    ax1.annotate(
        row['code_module'],
        (row['avg_course_length_days'], row['avg_completion_rate_pct']),
        fontsize=9, fontweight='bold', ha='center', va='bottom',
        xytext=(0, 8), textcoords='offset points',
    )
ax1.set_xlabel(DESIGN_LABELS['avg_course_length_days'])
ax1.set_ylabel(LABEL_COMPLETION_RATE)
ax1.set_title('Course Length vs Completion Rate')
sns.despine(ax=ax1)

# --- Right: assessment density vs completion ---
for i, (_, row) in enumerate(df_bq4.iterrows()):
    ax2.scatter(
        row['avg_assessment_density'], row['avg_completion_rate_pct'],
        color=PALETTE_COURSE[row['code_module']],
        s=size_scale[i], edgecolor='white', linewidth=1.5, zorder=3,
    )
    ax2.annotate(
        row['code_module'],
        (row['avg_assessment_density'], row['avg_completion_rate_pct']),
        fontsize=9, fontweight='bold', ha='center', va='bottom',
        xytext=(0, 8), textcoords='offset points',
    )
ax2.set_xlabel(DESIGN_LABELS['avg_assessment_density'])
ax2.set_ylabel(LABEL_COMPLETION_RATE)
ax2.set_title('Assessment Density vs Completion Rate')
sns.despine(ax=ax2)

fig.suptitle(
    'Course Design Features vs Completion Rate\n'
    '(bubble size = total enrollment)',
    fontsize=14, y=1.02,
)
fig.tight_layout()
save_fig(fig, '06_course_design_vs_completion')
plt.show()

> **Visual patterns:**
> - **Course length:** The scatter may suggest a relationship between duration and retention — shorter or longer courses may perform differently. However, with 7 points, any apparent trend could be driven by one or two outliers.
> - **Assessment density:** Courses with different assessment pacing may show different retention profiles. Whether more frequent assessment helps (through regular engagement checkpoints) or hurts (through assessment fatigue) is an open question.
>
> **Confounding factors:** These associations are confounded by subject difficulty, student self-selection, and module content quality. A course with high assessment density that also happens to be in a popular, well-taught subject will retain students regardless of assessment frequency.

In [ ]:
# --- Exploratory rank correlations ---
# With only 7 data points, formal significance testing is meaningless
# (critical Spearman ρ for n=7 at α=0.05 two-tailed ≈ 0.79).
# These correlations serve as exploratory pattern summaries, not evidence.
all_features = DESIGN_FEATURES + ENGAGEMENT_FEATURES
all_labels = {**DESIGN_LABELS, **ENGAGEMENT_LABELS}

rho_series = (
    df_bq4[all_features + ['avg_completion_rate_pct']]
    .corr(method='spearman')['avg_completion_rate_pct']
    .drop('avg_completion_rate_pct')
)

print('=== Exploratory Spearman ρ vs Completion Rate (n=7) ===')
print('  CAUTION: 7 data points → descriptive patterns only, not statistical evidence.')
print('  |ρ| > 0.79 needed for significance at α=0.05 (two-tailed).')
print()

for feat in all_features:
    rho = rho_series[feat]
    label = all_labels[feat]
    if abs(rho) >= 0.79:
        direction = '★ notable'
    elif abs(rho) >= 0.4:
        direction = '~ moderate'
    else:
        direction = '— weak'
    print(f'  {label:35s} ρ = {rho:+.3f}  ({direction})')

> **Correlation summary:** The Spearman rank correlations provide a compact summary of which features *move in the same direction* as completion rate. Features marked as notable (|ρ| ≥ 0.79) would pass the significance threshold for n=7, but even these should be treated as hypotheses.
>
> **What correlations cannot tell us:** Rank correlations measure monotonic association, not causation. A strong positive ρ between assessment density and completion rate could mean that assessments help retention — or that popular, well-designed courses happen to have more assessments. The data cannot distinguish these explanations with 7 observations.

## 5. Engagement Intensity by Course

Do students engage differently with different courses? Two complementary metrics capture engagement intensity:

- **Average clicks per student-day** (from `v_engagement_daily`): the typical daily VLE interaction intensity. Higher values mean students who log in tend to click more.
- **Median early clicks** (from `v_engagement_early`): the median total clicks in the first 28 days. This captures the initial engagement burst and is less sensitive to outliers than the mean.

The modules are sorted by completion rate (same order as the ranking chart) to facilitate visual comparison.

In [ ]:
# --- Engagement intensity by course ---
# 1x2 panel: daily engagement intensity + early engagement volume.
# Modules sorted by completion rate for consistent reading.
df_eng = df_bq4.sort_values(
    'avg_completion_rate_pct', ascending=True
).reset_index(drop=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=FIG_SIZE_WIDE)

y_pos = np.arange(len(df_eng))
colors = [PALETTE_COURSE[m] for m in df_eng['code_module']]

# --- Left: avg clicks per student-day ---
ax1.barh(y_pos, df_eng['avg_clicks_per_student_day'], color=colors, edgecolor='white')
for i, (_, row) in enumerate(df_eng.iterrows()):
    val = row['avg_clicks_per_student_day']
    ax1.text(
        val + 0.5, i,
        f'{val:.1f}',
        va='center', fontsize=9, color='#333333',
    )
ax1.set_yticks(y_pos)
ax1.set_yticklabels(df_eng['code_module'])
ax1.set_xlabel(ENGAGEMENT_LABELS['avg_clicks_per_student_day'])
ax1.set_title('Daily Engagement Intensity')
sns.despine(ax=ax1)

# --- Right: median early clicks (first 28 days) ---
ax2.barh(y_pos, df_eng['median_early_clicks'], color=colors, edgecolor='white')
for i, (_, row) in enumerate(df_eng.iterrows()):
    val = row['median_early_clicks']
    ax2.text(
        val + 10, i,
        f'{val:.0f}',
        va='center', fontsize=9, color='#333333',
    )
ax2.set_yticks(y_pos)
ax2.set_yticklabels(df_eng['code_module'])
ax2.set_xlabel(ENGAGEMENT_LABELS['median_early_clicks'])
ax2.set_title('Early Engagement Volume (first 28 days)')
sns.despine(ax=ax2)

fig.suptitle(
    'Engagement Intensity by Course\n'
    '(modules sorted by completion rate, bottom = lowest)',
    fontsize=14, y=1.02,
)
fig.tight_layout()
save_fig(fig, '06_engagement_by_course')
plt.show()

> **Engagement variation:** Engagement intensity varies across courses, but the relationship with completion is not straightforward. A course might have high per-session engagement (many clicks per visit) but low completion if students eventually disengage.
>
> **Two engagement dimensions:**
> - **Daily intensity** reflects how deeply students interact *when they log in*. This is influenced by course design: resource-heavy courses with interactive elements naturally generate more clicks.
> - **Early volume** reflects the initial engagement momentum. BQ2 (NB04) showed that early engagement is the strongest predictor of completion at the student level. Here we see whether some courses generate more initial engagement than others.
>
> **Interpretation caution:** Engagement metrics at the course level confound course design with student behavior. A course that attracts more motivated students will show higher engagement regardless of its design quality.

## 6. Course Design Heatmap

The heatmap standardizes all design and engagement features to **z-scores** (standard deviations from the 7-module mean) so that features on different scales become visually comparable.

- **Green cells** indicate above-average values (relative to other modules)
- **Red cells** indicate below-average values
- **Annotations** show the actual (unstandardized) values for reference

Modules are sorted top-to-bottom by completion rate, so visual patterns between the top (high-retention) and bottom (low-retention) courses become apparent.

In [ ]:
# --- Course design heatmap ---
# Z-score standardization puts all features on the same scale for visual
# comparison: how many standard deviations above or below the module mean.
all_features = DESIGN_FEATURES + ENGAGEMENT_FEATURES
all_labels = {**DESIGN_LABELS, **ENGAGEMENT_LABELS}

# Sort by completion rate: highest at top for intuitive reading
module_order = df_bq4.sort_values(
    'avg_completion_rate_pct', ascending=False
)['code_module'].tolist()

df_heat_raw = df_bq4.set_index('code_module').loc[module_order, all_features]

# Z-score: (value - mean) / std across the 7 modules for each feature
df_z = (df_heat_raw - df_heat_raw.mean()) / df_heat_raw.std()

# Readable column labels
readable_cols = [all_labels[c] for c in all_features]
df_z.columns = readable_cols

# Build annotation matrix with actual values (not z-scores)
# so the reader sees real numbers alongside the color encoding
annot_values = []
for _, row in df_heat_raw.iterrows():
    row_annot = []
    for feat in all_features:
        val = row[feat]
        if feat == 'avg_assessment_density':
            row_annot.append(f'{val:.2f}')
        elif feat in ('avg_n_assessments', 'avg_n_activity_types',
                       'avg_clicks_per_student_day'):
            row_annot.append(f'{val:.1f}')
        else:
            row_annot.append(f'{val:.0f}')
    annot_values.append(row_annot)

annot_array = np.array(annot_values)

# --- Heatmap ---
fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    df_z, annot=annot_array, fmt='',
    cmap='RdYlGn', center=0, linewidths=0.5,
    ax=ax, cbar_kws={'label': 'Z-score (standard deviations from mean)'},
)

# Add completion rate to y-axis labels for immediate context
rate_labels = []
for m in module_order:
    mask = df_bq4['code_module'] == m
    rate = df_bq4.loc[mask, 'avg_completion_rate_pct'].values[0]
    rate_labels.append(f'{m}  ({rate:.1f}%)')
ax.set_yticklabels(rate_labels, rotation=0)

ax.set_title(
    'Course Design Heatmap — All Features Standardized\n'
    '(z-score coloring, actual values annotated; sorted by completion rate)'
)
ax.set_ylabel('')
fig.tight_layout()
save_fig(fig, '06_course_design_heatmap')
plt.show()

> **Reading the heatmap:** Look for vertical color patterns (features that consistently differ between high- and low-retention courses) and horizontal patterns (modules with consistently above- or below-average characteristics).
>
> **What to look for:**
> - Do high-completion courses (top rows) tend to be greener on specific feature columns? If so, those features may contribute to retention.
> - Do low-completion courses (bottom rows) share red cells on the same features? That would strengthen the pattern.
> - Are there modules that are outliers — green on most features but low on completion, or vice versa? These break the pattern and suggest other factors at play.
>
> **Caveat:** The z-score standardization amplifies visual contrast — a difference of 0.5 standard deviations looks dramatic in the color scale but may not be practically meaningful with only 7 observations.

## 7. Hypotheses and Limitations

### Hypotheses (not conclusions)

Based on the descriptive patterns observed:

1. **Assessment structure may influence retention.** Courses with a certain assessment pacing might provide regular engagement checkpoints that help students stay on track — or might create pressure points that drive departures. The direction of this effect is an empirical question.

2. **Resource diversity may support engagement.** Courses offering a wider variety of VLE activity types may keep students engaged through varied learning experiences. However, resource diversity could also correlate with institutional investment in the course, which is a confound.

3. **Initial engagement intensity varies by course design.** Some courses generate higher early engagement, which NB04 (BQ2) identified as the strongest student-level predictor of completion. If course design can increase initial engagement, this is an actionable lever for retention.

### Limitations

This analysis has fundamental constraints that must be acknowledged:

- **n=7.** Seven data points cannot support inferential statistics. All correlations, patterns, and hypotheses are exploratory. A significant Spearman ρ at n=7 requires |ρ| > 0.79 — extremely strong associations.

- **Ecological fallacy.** Course-level averages can mask student-level variation. A course with high average engagement might have a bimodal distribution: many highly engaged students and many who never logged in.

- **Confounding by subject.** Different modules teach different subjects. Subject inherent difficulty, career relevance, and student motivation are unobserved variables that could explain both course design choices and retention outcomes.

- **Student self-selection.** Students choose which modules to take. If more capable or motivated students select certain courses, those courses will show higher retention regardless of design.

- **Temporal confounding.** Course design may have changed across presentations, and aggregate metrics smooth over these changes.

## 8. Key Takeaways and Next Steps

### What we learned

1. **Completion rates vary substantially across modules** — the gap between best and worst performers is meaningful and consistent across presentations. This variation is not noise.

2. **Course design features show suggestive patterns** with retention, but with only 7 observations, these are hypotheses rather than findings. Assessment density and resource diversity emerge as candidates for further investigation.

3. **Engagement intensity differs by course.** Some modules generate more VLE interaction than others. Since early engagement is the strongest student-level predictor of completion (BQ2), course designs that promote higher initial engagement may indirectly support retention.

4. **The heatmap reveals course profiles** — modules differ not just in completion rate but in their overall design and engagement characteristics. High-retention courses may share certain design patterns.

5. **No causal claims are possible** at this sample size. The patterns observed here are inputs to the recommendations in BQ5, not standalone evidence.

### What comes next

| Notebook | Business Question | Focus |
|----------|-------------------|-------------------------------------------------|
| **07** | BQ5 | Top 3 actionable interventions for retention |

NB07 synthesizes findings from all five business questions (BQ1–BQ4) into concrete, prioritized recommendations for a platform operator — the final analytical output of this project.

---

**Reproducibility:** All figures are saved to `reports/figures/`. To reproduce this notebook, run `python -m run_pipeline` first, then execute all cells in order.

> **From course profiles to action:** This notebook characterized *how courses differ* in design, engagement, and retention. Combined with the student-level insights from BQ1–BQ3, we now have the full picture needed to answer the final question: what can a platform operator actually do?
>
> Continue to **Notebook 07** (`07_bq5_recommendations_synthesis.ipynb`) for BQ5: the top 3 actionable interventions for improving student retention.